# Parte 1 — Ingestão e integração

Leitura dos três CSVs da PRF, padronização dos nomes de coluna, conferência de schema
e união numa base única, com prova de que nenhuma linha foi perdida.


## 1.1 Setup


In [1]:
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd


def localizar_raiz() -> Path:
    for candidato in [Path.cwd(), *Path.cwd().parents]:
        if (candidato / "dados_brutos").is_dir():
            return candidato
    raise FileNotFoundError("Abra o notebook da raiz do projeto ou de 'notebooks/'.")


RAIZ = localizar_raiz()
DIR_BRUTOS = RAIZ / "dados_brutos"
DIR_TRATADOS = RAIZ / "dados_tratados"
DIR_RESULTADOS = RAIZ / "resultados"
for pasta in (DIR_TRATADOS, DIR_RESULTADOS):
    pasta.mkdir(parents=True, exist_ok=True)

SEPARADOR = ";"
ENCODING_SAIDA = "utf-8-sig"
ANOS = (2023, 2024, 2025)
print(RAIZ)


/home/user/PRF-pandas-e-Plotly---2023-2025


## 1.2 Leitura dos três CSVs (`ler_csv_prf` com fallback de encoding)

`utf-8` falha nos três arquivos da PRF, então o fallback para `latin1` é realmente usado.
`id` é lido como texto porque em 2024 há uma linha com `id = "6e+05"` (veio de planilha).


In [2]:
def ler_csv_prf(caminho: Path) -> pd.DataFrame:
    """Lê o CSV tentando vários encodings; só falha se todos falharem."""
    for encoding in ("utf-8", "latin1", "utf-8-sig", "cp1252"):
        try:
            df = pd.read_csv(caminho, sep=SEPARADOR, encoding=encoding,
                             low_memory=False, dtype={"id": "string"})
        except (UnicodeDecodeError, UnicodeError):
            continue
        print(f"{caminho.name}: encoding={encoding}, {len(df):,} linhas")
        return df
    raise RuntimeError(f"Nenhum encoding funcionou para {caminho}")


def normalizar_nome(nome: str) -> str:
    nome = str(nome).strip().lower()
    nome = unicodedata.normalize("NFKD", nome).encode("ascii", "ignore").decode()
    return nome.replace(" ", "_").replace("-", "_").replace("/", "_").strip("_")


In [3]:
dados_por_ano = {ano: ler_csv_prf(DIR_BRUTOS / f"datatran{ano}.csv") for ano in ANOS}


datatran2023.csv: encoding=latin1, 67,766 linhas


datatran2024.csv: encoding=latin1, 73,202 linhas


datatran2025.csv: encoding=latin1, 72,529 linhas


## 1.3 Padronização de colunas e conferência entre os anos


In [4]:
for df in dados_por_ano.values():
    df.columns = [normalizar_nome(c) for c in df.columns]

colunas = {ano: set(df.columns) for ano, df in dados_por_ano.items()}
assert set.intersection(*colunas.values()) == set.union(*colunas.values()), \
    "Há coluna que não existe em todos os anos"
print(f"Colunas: {sorted(colunas[2023])}")


Colunas: ['br', 'causa_acidente', 'classificacao_acidente', 'condicao_metereologica', 'data_inversa', 'delegacia', 'dia_semana', 'fase_dia', 'feridos', 'feridos_graves', 'feridos_leves', 'horario', 'id', 'ignorados', 'ilesos', 'km', 'latitude', 'longitude', 'mortos', 'municipio', 'pessoas', 'regional', 'sentido_via', 'tipo_acidente', 'tipo_pista', 'tracado_via', 'uf', 'uop', 'uso_solo', 'veiculos']


## 1.4 `arquivo_origem` e `concat` em `acidentes_prf_historico`


In [5]:
for ano, df in dados_por_ano.items():
    df["arquivo_origem"] = ano

acidentes_prf_historico = pd.concat(
    [dados_por_ano[ano] for ano in ANOS], ignore_index=True
)
print(acidentes_prf_historico.shape)


(213497, 31)


## 1.5 `assert` do total de linhas

Verificado **antes** da remoção de duplicatas da Parte 2.


In [6]:
soma = sum(len(df) for df in dados_por_ano.values())
assert len(acidentes_prf_historico) == soma
print(f"{soma:,} linhas = 67.766 + 73.202 + 72.529")
print(f"Fatais: {int((acidentes_prf_historico['mortos'] >= 1).sum()):,}")


213,497 linhas = 67.766 + 73.202 + 72.529
Fatais: 15,293
